In [1]:
import numpy as np
from random import randint
import random

class EnvGri(object):
    def __init__(self):
        super(EnvGri, self).__init__()
        # la grille 4x4 : 0=case libre, -1=mur, -100=piège, 100=sortie
        self.grid=[
            [0,0,0,0],
            [0,-1,0,-100],
            [0,0,-1,0],
            [-100,0,0,100],
        ]
        self.y=0  # position ligne de l'agent (0=haut)
        self.x=0  # position colonne de l'agent (0=gauche)
        # chaque action = [delta_y, delta_x]
        self.action=[
            [-1,0],  # 0 = haut
            [1,0],   # 1 = bas
            [0,-1],  # 2 = gauche
            [0,1],   # 3 = droite
        ]

    def reset(self):
        self.y=0  # replacer l'agent au départ (ligne 0)
        self.x=0  # replacer l'agent au départ (colonne 0)
        return 0  # retourner l'état initial = index 0

    def step(self, action):
        # déplacer y en restant dans les bornes 0 à 3
        self.y = max(0, min(self.y + self.action[action][0], 3))
        # déplacer x en restant dans les bornes 0 à 3
        self.x = max(0, min(self.x + self.action[action][1], 3))

        reward = self.grid[self.y][self.x]           # lire la récompense de la case
        done = reward == 100 or reward == -100        # épisode terminé si sortie ou piège
        state = self.y * 4 + self.x                  # convertir (y,x) en index 0-15
        return state, reward, done

    def isfinished(self):
        # vrai si l'agent est sur la case sortie (valeur 100)
        return self.grid[self.y][self.x]==100

    def show(self):
        # afficher la grille avec X = position actuelle de l'agent
        print("-----------------")
        y=0
        for line in self.grid:
            x=0
            for pt in line:
                # afficher X si c'est la position de l'agent, sinon la valeur de la case
                print("%s\t"%(pt if y!=self.y or x!=self.x else "X"),end="")
                x=x+1
            print("")  # saut de ligne après chaque ligne de la grille
            y=y+1      # passer à la ligne suivante

    def show_best_moves(self, Q):
        arrows = ['↑', '↓', '←', '→']  # symboles des 4 actions
        print("\nMeilleure action par état :")
        print("-" * 25)
        for y in range(4):
            for x in range(4):
                state = y * 4 + x          # convertir (y,x) en index état
                val = self.grid[y][x]      # lire la valeur de la case
                if val == 100:
                    print(" ★ ", end="")   # case sortie
                elif val == -100:
                    print(" ✕ ", end="")   # case piège
                elif val == -1:
                    print(" ▪ ", end="")   # case mur
                elif all(v == 0 for v in Q[state]):
                    print(" ? ", end="")   # état jamais visité
                else:
                    best = int(np.argmax(Q[state]))    # action avec la valeur Q max
                    print(f" {arrows[best]} ", end="") # afficher la flèche correspondante
            print("")  # saut de ligne après chaque ligne
        print("-" * 25)

# ================================================================
# AGENT : choisir une action selon la stratégie epsilon-greedy
# ================================================================
def take_action(st, q_table, eps):
    if random.uniform(0, 1) < eps:   # tirer un nombre entre 0 et 1
        action = randint(0, 3)        # exploration : action aléatoire
    else:
        action = np.argmax(q_table[st])  # exploitation : meilleure action connue
    return action

# ================================================================
# HYPERPARAMÈTRES
# ================================================================
alpha     = 0.1    # taux d'apprentissage : à quel point on met à jour la Q-table
gamma     = 0.99   # facteur de discount : importance des récompenses futures
eps       = 1.0    # epsilon initial : 100% exploration au départ
eps_min   = 0.01   # epsilon minimum : on explore toujours un peu
eps_decay = 0.995  # facteur de décroissance d'epsilon après chaque épisode

# ================================================================
# INITIALISATION
# ================================================================
env = EnvGri()          # créer l'environnement
Q   = np.zeros((16, 4)) # Q-table vide : 16 états x 4 actions = tout à 0

# ================================================================
# BOUCLE D'ENTRAÎNEMENT
# ================================================================
for episode in range(1000):       # répéter 1000 épisodes
    st = env.reset()              # remettre l'agent au départ, récupérer état initial
    while True:
        at = take_action(st, Q, eps)       # choisir une action (exploration ou exploitation)
        stp1, r, done = env.step(at)       # exécuter l'action, récupérer nouvel état, récompense, fin

        # mise à jour Q-table avec la formule Q-learning :
        # Q(s,a) = Q(s,a) + alpha * [ r + gamma * max(Q(s+1)) - Q(s,a) ]
        Q[st][at] = Q[st][at] + alpha * (r + gamma * np.max(Q[stp1]) - Q[st][at])

        st = stp1   # avancer vers le nouvel état
        if done:    # si épisode terminé (sortie ou piège), passer au suivant
            break

    # réduire epsilon après chaque épisode pour explorer de moins en moins
    eps = max(eps_min, eps * eps_decay)

# ================================================================
# RÉSULTATS
# ================================================================
print("\nQ-table finale :")
for s in range(16):
    print(s, Q[s])   # afficher les valeurs Q de chaque état

env.show_best_moves(Q)  # afficher la grille avec les meilleures actions apprises


Q-table finale :
0 [91.59303508 86.42708872 88.73478304 94.12870599]
1 [92.70594752 90.27250822 91.53891547 95.079501  ]
2 [91.76821329 96.0399     92.1291122  86.22677918]
3 [  9.14715484 -89.05810109  93.3248759   14.08928891]
4 [91.75276291 12.86148576 42.63076074 42.05406924]
5 [56.4744896  48.69997035 40.08076467 95.92539372]
6 [ 89.42372175  97.01        89.38244006 -99.93038014]
7 [0. 0. 0. 0.]
8 [  0.56954729 -90.15229098   0.104558    52.78565156]
9 [27.21121927 41.73973444 10.73992028 96.00577225]
10 [92.97732072 99.         88.59155259 80.59864983]
11 [-46.8559      94.1850263   50.91270212  21.29390581]
12 [0. 0. 0. 0.]
13 [ 24.8089607   22.29461469 -79.41088679  94.96496882]
14 [ 94.64889129  95.46458514  81.84617004 100.        ]
15 [0. 0. 0. 0.]

Meilleure action par état :
-------------------------
 →  →  ↓  ← 
 ↑  ▪  ↓  ✕ 
 →  →  ▪  ↓ 
 ✕  →  →  ★ 
-------------------------
